# Cleaning DSLD labels

Repository gets the raw records, the service cleans and flattens them, the
repository saves the result. The service itself touches no files.

Each cell is one call.

In [9]:
from engine.repositories.dsld import loader, store
from engine.services import dsld_cleaning

## Get

In [10]:
raw_labels = loader.load_raw(n=20, order="random", seed=7)
len(raw_labels)

20

In [12]:
# What a raw record carries -- most of it is not label text.
# sorted(raw_labels[0])
raw_labels[0]



{'src': '01-raw/2025-09-25/data/label_17.json',
 'id': 202878,
 'nhanesId': '',
 'bundleName': '',
 'fullName': 'Advanced Brain',
 'brandName': 'New Sun',
 'brandIpSymbol': '™',
 'upcSku': '48-4',
 'productVersionCode': 'Product 48-4 v7.0',
 'pdf': '',
 'thumbnail': '',
 'servingsPerContainer': '30',
 'hasOuterCarton': False,
 'percentDvFootnote': 'Not Present',
 'labelRelationships': [{'type': 'Image difference, same product',
   'labelId': 82157},
  {'type': 'Image difference, same product', 'labelId': 11675},
  {'type': 'Image difference, same product', 'labelId': 244890}],
 'contacts': [{'contactId': 9770,
   'text': 'Manufactured for ',
   'types': ['Distributor'],
   'contactDetails': {'src': '01-raw/2025-09-25/data/contact_3.json',
    'id': 9770,
    'name': 'New Sun, Inc.',
    'streetAddress': '',
    'city': 'Hendersonville',
    'state': 'NC',
    'country': '',
    'zipCode': '28792',
    'phoneNumber': '(800) 544-0777',
    'email': '',
    'webAddress': 'mynewsun.com'}}]

## Clean

Four paths are read: `statements[i].notes`, and `notes`/`description` on every
ingredient row, descending `nestedRows`. Everything else is absent because
nothing reads it.

In [13]:
cleaned = dsld_cleaning.clean(raw_labels)
cleaned[0]

CleanedLabel(dsld_id=202878, name='Advanced Brain', segments=[Segment(text='Recommendation: As a dietary supplement, take one capsule one to two times daily. Do not exceed dosage without advice of a physician.', paths=['statements[0].notes']), Segment(text='Do not exceed dosage without advice of a physician.', paths=['statements[1].notes']), Segment(text='Do not use if safety seal is broken or missing.', paths=['statements[2].notes']), Segment(text='Keep out of reach of children.', paths=['statements[3].notes']), Segment(text='Store in a cool, dry place.', paths=['statements[4].notes']), Segment(text='Contains soy Lecithin', paths=['statements[5].notes']), Segment(text='Allergen information: Contains soy lecithin.', paths=['statements[6].notes']), Segment(text='Warning: Do not use if you are pregnant or nursing. Do not take this product if you are taking blood-thinning or cholinergic medications without first consulting a doctor. If you are taking any medications, have a history of hea

In [5]:
# Each surviving piece of text remembers every place it came from.
for segment in cleaned[0].segments[:5]:
    print(segment.paths, segment.text[:70])

['statements[0].notes'] Recommendation: As a dietary supplement, take one capsule one to two t
['statements[1].notes'] Do not exceed dosage without advice of a physician.
['statements[2].notes'] Do not use if safety seal is broken or missing.
['statements[3].notes'] Keep out of reach of children.
['statements[4].notes'] Store in a cool, dry place.


In [6]:
# Identical text under several statements merges, keeping both paths.
multi = [s for label in cleaned for s in label.segments if len(s.paths) > 1]
multi[:3]

[Segment(text='Contains: Fish (cod, cusk, flounder, haddock, hake, pollock, redfish, sole). Contains FD&C Yellow no. 5 Lake - Tartrazine as a color additive.', paths=['statements[14].notes', 'statements[15].notes']),
 Segment(text='Pregnant or nursing mothers, children under the age of 18 and individuals with a known medical condition should consult a physician before using this or any dietary supplement.', paths=['statements[3].notes', 'statements[4].notes']),
 Segment(text='Pregnant or nursing mothers, children under 18, and individuals with a known medical condition should consult a physician before using this or any dietary supplement.', paths=['statements[3].notes', 'statements[15].notes'])]

## Flatten

One row per label. `passage` is what a prompt sends; `segments` rides along as
provenance, and `renderer_version` says which renderer produced the text.

In [7]:
rows = dsld_cleaning.flatten(cleaned)
{k: v for k, v in rows[0].items() if k != "segments"}

{'dsld_id': 202878,
 'name': 'Advanced Brain',
 'passage': 'Recommendation: As a dietary supplement, take one capsule one to two times daily. Do not exceed dosage without advice of a physician.\nDo not exceed dosage without advice of a physician.\nDo not use if safety seal is broken or missing.\nKeep out of reach of children.\nStore in a cool, dry place.\nContains soy Lecithin\nAllergen information: Contains soy lecithin.\nWarning: Do not use if you are pregnant or nursing. Do not take this product if you are taking blood-thinning or cholinergic medications without first consulting a doctor. If you are taking any medications, have a history of heart or pulmonary problems, or experiencing health issues, consult your doctor before use. May potentiate pharmaceutical MAO inhibitors.\nThis product is not intended for long term use.\nThis statement has not been evaluated by the Food and Drug Administration. This product is not intended to diagnose, treat, cure, or prevent any disease.\nProdu

In [8]:
print(rows[0]["passage"][:600])

Recommendation: As a dietary supplement, take one capsule one to two times daily. Do not exceed dosage without advice of a physician.
Do not exceed dosage without advice of a physician.
Do not use if safety seal is broken or missing.
Keep out of reach of children.
Store in a cool, dry place.
Contains soy Lecithin
Allergen information: Contains soy lecithin.


In [9]:
# The product name is carried on the row and kept out of the passage: grounding
# is a substring test, so text the label never stated must not become quotable.
rows[0]["name"], rows[0]["name"] in rows[0]["passage"]

('Advanced Brain', False)

## Save

In [10]:
run_dir = store.run_dir()
manifest = {
    "task": "dsld_cleaning",
    "renderer_version": rows[0]["renderer_version"],
    "n_labels": len(rows),
    "n_empty_passages": sum(1 for r in rows if not r["passage"]),
    "dsld_ids": [r["dsld_id"] for r in rows],
    "draw": {"n": 20, "order": "random", "seed": 7},
}
written = store.write_passages(run_dir, rows, manifest)
written

{'rows': PosixPath('/var/tmp/ge74wan/repos/supplement-kg/data/dsld/runs/20260921T141035Z/raw/passages.jsonl'),
 'manifest': PosixPath('/var/tmp/ge74wan/repos/supplement-kg/data/dsld/runs/20260921T141035Z/manifests/passages.json')}

In [11]:
print(written["rows"].read_text().splitlines()[0][:300])

{"dsld_id": 202878, "name": "Advanced Brain", "passage": "Recommendation: As a dietary supplement, take one capsule one to two times daily. Do not exceed dosage without advice of a physician.\nDo not exceed dosage without advice of a physician.\nDo not use if safety seal is broken or missing.\nKeep 


## How much the cleaning removes

Characters of raw JSON against characters of passage, over the drawn labels.

In [12]:
import json

raw_chars = sum(len(json.dumps(r)) for r in raw_labels)
passage_chars = sum(len(r["passage"]) for r in rows)
print(f"raw {raw_chars:,} -> passage {passage_chars:,} "
      f"({100 * (1 - passage_chars / raw_chars):.1f}% smaller)")

raw 225,561 -> passage 31,364 (86.1% smaller)


---

## The 1,000-label silver set

Not run by default -- it reads a thousand files from the dump and takes a couple
of minutes. It writes to a **fresh** run directory: `silver1000-text-v2/raw/inputs.jsonl`
is what `merge_silver1000.py` validates its shards against and must not be overwritten.

`tests/test_dsld_cleaning.py` already asserts this service reproduces the audited
renderer byte for byte across both the gold and silver sets; this cell is for
producing the file, not for checking equivalence.

In [13]:
# from pathlib import Path
#
# ids = [json.loads(line)["dsld_id"]
#        for line in Path("../datasets/pilot/silver-1000.jsonl").read_text().splitlines()
#        if line.strip()]
#
# silver_rows = dsld_cleaning.flatten(dsld_cleaning.clean(loader.load_raw(ids)))
# silver_dir = store.run_dir()
# store.write_passages(silver_dir, silver_rows, {
#     "task": "dsld_cleaning",
#     "source": "silver-1000",
#     "renderer_version": silver_rows[0]["renderer_version"],
#     "n_labels": len(silver_rows),
# })